# Tiny-array practice for clustering (numpy only)

These master-level projects are the hardest so far, and the gap is rarely the math — it is array shapes plus a few numpy tricks. So here everything is tiny and visible: the same X (3 points, 2 numbers each) and C (2 centroids) all the way down, with shapes printed at every step.

Quick reminder used everywhere below: axis=0 collapses the rows, leaving one number per column. Each exercise is one trick only — cover the solution cell, try the few lines, then peek.

In [29]:
import numpy as np

X = np.array([[0., 0.], [1., 1.], [2., 0.]])
C = np.array([[0., 0.], [1., 0.]])
# print("X has", X.shape[0], "rows", X.shape[1],"cols \n", C.shape)
print(X, "\n X has", X.shape[0], "rows", X.shape[1],"cols")
print(X.min(axis=0), " return minimum along axis")
print(X.mean(axis=1), " return avg along axis")

[[0. 0.]
 [1. 1.]
 [2. 0.]] 
 X has 3 rows 2 cols
[0. 0.]  return minimum along axis
[0. 1. 1.]  return avg along axis


## 1 — Slicing reminder: `:` keeps, `None` adds

Start from X with shape (3, 2): 3 points, 2 numbers each.

`:` means "take everything in this slot". `None` (written `np.newaxis` in project code) adds a brand-new slot of size 1 — and where you slip it in decides the new shape.

Your turn: carve out the first column, then land the new slot in three different spots: (3, 1, 2) vs (1, 3, 2) vs (3, 2, 1).

In [40]:
# your turn: ':' keeps a slot, None adds a size-1 slot
col = X[0]
col2 = X[:,0] # notice difference
print(col.shape, col2.shape)
print(col,col2)

(2,) (3,)
[0. 0.] [0. 1. 2.]
(3, 1, 2) (3, 1, 2)
[[[0. 0.]]

 [[1. 1.]]

 [[2. 0.]]] 
 [[[0. 0.]]

 [[1. 1.]]

 [[2. 0.]]]


In [46]:
# a and b are X with a new empty dim - play around with slice
a = X[:, np.newaxis]
b = X[:, None]
print(a.shape, b.shape)
print(a,"\n ---- \n",b)

IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed

In [47]:
# solution
col = X[:, 0]
a = X[:, np.newaxis, :]
b = X[np.newaxis, :, :]
c = X[:, :, np.newaxis]
print(col.shape, a.shape, b.shape, c.shape)
print(col)

(3,) (3, 1, 2) (1, 3, 2) (3, 2, 1)
[0. 1. 2.]


## 2 — One subtraction for every pair

Picture 3 points, 2 centroids, 2 coords — you want every point compared with every centroid in a single move.

Think of diffs as a table of arrows: one row per point, one column per centroid, each cell holding a 2-number arrow.

Your turn: build diffs with shape (3, 2, 2), then point at one cell like diffs[2, 0] and say in plain words which point and which centroid that arrow connects.

In [ ]:
# your turn: one arrow per point-centroid pair, no loops
diffs = X  # TODO -> (3, 2, 2)
print(diffs.shape)
print(diffs)

In [48]:
# solution: diffs[i, j] is the arrow from centroid j to point i
diffs = X[:, np.newaxis, :] - C
print(diffs.shape)
print(diffs)

(3, 2, 2)
[[[ 0.  0.]
  [-1.  0.]]

 [[ 1.  1.]
  [ 0.  1.]]

 [[ 2.  0.]
  [ 1.  0.]]]


## 3 — Closest centroid per point

diffs is still (3, 2, 2): an arrow per point-centroid pair.

First turn each arrow into a single number by adding up its coords, then give every point to its nearest centroid — one label per row.

Your turn: make sq with shape (3, 2) and clss with shape (3,), and read clss out loud like "point 0 goes to centroid …".

In [ ]:
# your turn: arrows -> one number per pair -> one label per point
sq = diffs  # TODO -> (3, 2)
clss = sq  # TODO -> (3,)
print(sq.shape, clss.shape)
print(clss)

In [ ]:
# solution
sq = (diffs ** 2).sum(axis=2)
clss = sq.argmin(axis=1)
print(sq.shape, clss.shape)
print(sq)
print(clss)

## 4 — Total spread in two moves

sq is (3, 2): one squared number per point-centroid pair, no square roots involved.

Keep only each point's nearest number, then melt those winners down into a single total for the whole dataset.

Your turn: make nearest with shape (3,) and var, which is just one plain number.

In [ ]:
# your turn: nearest per point, then one total number
nearest = sq  # TODO -> (3,)
var = 0.0  # TODO -> one number
print(nearest.shape, var)
print(nearest)

In [ ]:
# solution
nearest = sq.min(axis=1)
var = nearest.sum()
print(nearest.shape, var)
print(nearest)

## 5 — A centroid is the average of its members

clss came out [0, 1, 1]: point 0 sits alone in cluster 0, points 1 and 2 share cluster 1, and cluster 2 got zero votes.

Rebuild one centroid by averaging only the points that voted for it — and count heads before averaging, because averaging an empty group is nonsense.

Your turn: rebuild the cluster-1 centroid with shape (2,) and flag whether cluster 2 is empty.

In [ ]:
# your turn: pick cluster-1 points, average them; check cluster 2
m1 = X  # TODO -> (2,)
empty = False  # TODO -> True or False?
print(m1.shape, empty)
print(m1)

In [ ]:
# solution
m1 = X[clss == 1].mean(axis=0)
empty = (clss == 2).sum() == 0
print(X[clss == 1].shape, m1.shape, empty)
print(m1)

## 6 — Split votes instead of winner-takes-all

Tiny toy, plain strengths: 2 groups, 3 points, one 2x3 table L plus a 2-number weighting pi.

Hard voting crowns one winner per point; soft voting splits each point's loyalty so that every column of g adds up to exactly 1.

Your turn: weigh the rows, then divide each column by its own total to get g with shape (2, 3).

In [ ]:
# your turn: weigh by pi, then split each column so it adds to 1
pi = np.array([0.5, 0.5])
L = np.array([[0.4, 0.3, 0.01], [0.1, 0.1, 0.4]])
g = L  # TODO -> (2, 3), columns add to 1
print(g.shape, g.sum(axis=0))
print(g)

In [ ]:
# solution
pi = np.array([0.5, 0.5])
L = np.array([[0.4, 0.3, 0.01], [0.1, 0.1, 0.4]])
unnorm = pi[:, None] * L
g = unnorm / unnorm.sum(axis=0)
print(g.shape, g.sum(axis=0))
print(g)